In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("netflix_customer_churn.csv")

In [3]:
df

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,44f3ba44-b95d-4e50-a786-bac4d06f4a43,19,Female,Basic,49.17,11,Europe,Desktop,8.99,0,Credit Card,4,4.10,Drama
4996,18779bcb-ba2b-41da-b751-e70b812061ec,67,Female,Basic,9.24,2,North America,Desktop,8.99,0,PayPal,3,3.08,Documentary
4997,3f32e8c5-615b-4a3b-a864-db2688f7834f,66,Male,Standard,16.55,49,South America,Desktop,13.99,1,Debit Card,2,0.33,Action
4998,7b0ad82d-6571-430e-90f4-906259e0e89c,59,Female,Basic,9.12,3,Europe,Laptop,8.99,0,Credit Card,4,2.28,Sci-Fi


In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow import keras
from tensorflow.keras import layers

import joblib

/Users/apple/Desktop/mission D/deep_learning/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [5]:
df.drop(columns=['customer_id', 'name'], inplace=True, errors='ignore')

df.dropna(inplace=True)

In [6]:
df = pd.get_dummies(
    df,
    columns=['gender','device','payment_method','favorite_genre','subscription_type','region'],
    drop_first=True
)


In [7]:
df.dtypes

age                             int64
watch_hours                   float64
last_login_days                 int64
monthly_fee                   float64
churned                         int64
number_of_profiles              int64
avg_watch_time_per_day        float64
gender_Male                      bool
gender_Other                     bool
device_Laptop                    bool
device_Mobile                    bool
device_TV                        bool
device_Tablet                    bool
payment_method_Crypto            bool
payment_method_Debit Card        bool
payment_method_Gift Card         bool
payment_method_PayPal            bool
favorite_genre_Comedy            bool
favorite_genre_Documentary       bool
favorite_genre_Drama             bool
favorite_genre_Horror            bool
favorite_genre_Romance           bool
favorite_genre_Sci-Fi            bool
subscription_type_Premium        bool
subscription_type_Standard       bool
region_Asia                      bool
region_Europ

In [8]:
df = df.astype(int)

In [9]:
df.dtypes

age                           int64
watch_hours                   int64
last_login_days               int64
monthly_fee                   int64
churned                       int64
number_of_profiles            int64
avg_watch_time_per_day        int64
gender_Male                   int64
gender_Other                  int64
device_Laptop                 int64
device_Mobile                 int64
device_TV                     int64
device_Tablet                 int64
payment_method_Crypto         int64
payment_method_Debit Card     int64
payment_method_Gift Card      int64
payment_method_PayPal         int64
favorite_genre_Comedy         int64
favorite_genre_Documentary    int64
favorite_genre_Drama          int64
favorite_genre_Horror         int64
favorite_genre_Romance        int64
favorite_genre_Sci-Fi         int64
subscription_type_Premium     int64
subscription_type_Standard    int64
region_Asia                   int64
region_Europe                 int64
region_North America        

In [10]:
X = df.drop('churned', axis=1)
y = df['churned']

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
X_train.shape

(4000, 29)

In [134]:
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)   # FULL dataset pe fit

# SAVE SCALER
joblib.dump(scaler, "scaler.pkl")


model = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid')
])

/Users/apple/Desktop/mission D/deep_learning/venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [135]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5)

In [136]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [137]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5519 - loss: 0.9257 - val_accuracy: 0.8525 - val_loss: 0.3587
Epoch 2/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7843 - loss: 0.4522 - val_accuracy: 0.8800 - val_loss: 0.2866
Epoch 3/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8351 - loss: 0.3581 - val_accuracy: 0.8800 - val_loss: 0.2641
Epoch 4/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8431 - loss: 0.3487 - val_accuracy: 0.8737 - val_loss: 0.2660
Epoch 5/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8550 - loss: 0.3198 - val_accuracy: 0.8737 - val_loss: 0.2653
Epoch 6/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8656 - loss: 0.2948 - val_accuracy: 0.8750 - val_loss: 0.2720
Epoch 7/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8628 - loss: 0.3104 - val_accuracy: 0.8700 - val_loss: 0.2623
Epoch 8/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8589 - loss: 0.3200 - val_accu

In [59]:
y_pred = model.predict(X_test)
print(y_pred[:20])

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
[[0.65592986]
 [0.80774224]
 [0.01237973]
 [0.84671307]
 [0.8382051 ]
 [0.99831283]
 [0.775236  ]
 [0.9970662 ]
 [0.50441444]
 [0.21402252]
 [0.06527565]
 [0.19128792]
 [0.04894833]
 [0.98131394]
 [0.6282138 ]
 [0.00151204]
 [0.91028136]
 [0.07871364]
 [0.24706437]
 [0.480017  ]]


In [70]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_binary)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[431  67]
 [ 43 459]]


In [71]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_binary))

              precision    recall  f1-score   support

           0       0.91      0.87      0.89       498
           1       0.87      0.91      0.89       502

    accuracy                           0.89      1000
   macro avg       0.89      0.89      0.89      1000
weighted avg       0.89      0.89      0.89      1000



In [89]:
y_pred = model.predict(X_test)

print("Min:", y_pred.min())
print("Max:", y_pred.max())

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Min: 1.2936706e-13
Max: 0.9999863


In [121]:
import numpy as np

pos = np.where(y_test.values == 1)[0][0]
sample = X_test[pos].reshape(1, -1)

prob = float(model.predict(sample, verbose=0)[0][0])

print(prob)

0.8077422380447388


In [138]:
import joblib

scaler = joblib.load("scaler.pkl")

In [139]:

data = {
    'age': 51,
    'watch_hours': 20,
    'last_login_days': 20,
    'monthly_fee': 199,   # FIXED
    'number_of_profiles': 1,
    'avg_watch_time_per_day': 0.5,  # FIXED

    'gender_Male': 0,
    'gender_Other': 1,

    'device_Laptop': 0,
    'device_Mobile': 1,
    'device_TV': 0,
    'device_Tablet': 0,

    'payment_method_Crypto': 0,
    'payment_method_Debit Card': 1,
    'payment_method_Gift Card': 0,
    'payment_method_PayPal': 0,

    'favorite_genre_Comedy': 0,
    'favorite_genre_Documentary': 0,
    'favorite_genre_Drama': 1,
    'favorite_genre_Horror': 0,
    'favorite_genre_Romance': 0,
    'favorite_genre_Sci-Fi': 0,

    'subscription_type_Premium': 0,
    'subscription_type_Standard': 1,

    'region_Asia': 1,   # IMPORTANT: ek hi 1 hona chahiye
    'region_Europe': 0,
    'region_North America': 0,
    'region_Oceania': 0,
    'region_South America': 0
}

In [141]:
import pandas as pd
import numpy as np

# ====== 1. TUMHARA DATA ======
data = {
    'age': 51,
    'watch_hours': 14,
    'last_login_days': 29,
    'monthly_fee': 199,  # FIXED (8 mat rakhna)
    'number_of_profiles': 1,
    'avg_watch_time_per_day': 0.5,

    'gender_Male': 0,
    'gender_Other': 1,

    'device_Laptop': 0,
    'device_Mobile': 1,
    'device_TV': 0,
    'device_Tablet': 0,

    'payment_method_Crypto': 0,
    'payment_method_Debit Card': 1,
    'payment_method_Gift Card': 0,
    'payment_method_PayPal': 0,

    'favorite_genre_Comedy': 0,
    'favorite_genre_Documentary': 0,
    'favorite_genre_Drama': 1,
    'favorite_genre_Horror': 0,
    'favorite_genre_Romance': 0,
    'favorite_genre_Sci-Fi': 0,

    'subscription_type_Premium': 0,
    'subscription_type_Standard': 1,

    'region_Asia': 1,
    'region_Europe': 0,
    'region_North America': 0,
    'region_Oceania': 0,
    'region_South America': 0
}

# ====== 2. DataFrame ======
new_user = pd.DataFrame([data])

# ====== 3. COLUMN CHECK ======
print("Columns match:", new_user.columns.equals(columns))

# ====== 4. MISSING VALUES ======
print("\nMissing values:\n", new_user.isnull().sum())

# ====== 5. ONE-HOT VALIDATION ======
def check_one_hot(cols, name):
    total = new_user[cols].sum(axis=1).values[0]
    print(f"{name} sum:", total)

check_one_hot(['subscription_type_Premium','subscription_type_Standard'], "Subscription")
check_one_hot(['region_Asia','region_Europe','region_North America','region_Oceania','region_South America'], "Region")
check_one_hot(['device_Laptop','device_Mobile','device_TV','device_Tablet'], "Device")

# ====== 6. ALIGN COLUMNS ======
new_user = new_user[columns]

# ====== 7. SCALE ======
new_user_scaled = scaler.transform(new_user)

print("\nScaled input:\n", new_user_scaled)

# ====== 8. PREDICT ======
prediction = model.predict(new_user_scaled, verbose=0)
prob = float(prediction[0][0])

# ====== 9. RESULT ======
if prob > 0.7:
    result = "High Risk (Churn)"
elif prob > 0.4:
    result = "Medium Risk"
else:
    result = "Low Risk (Stay)"

print("\nProbability:", prob)
print("Result:", result)

Columns match: True

Missing values:
 age                           0
watch_hours                   0
last_login_days               0
monthly_fee                   0
number_of_profiles            0
avg_watch_time_per_day        0
gender_Male                   0
gender_Other                  0
device_Laptop                 0
device_Mobile                 0
device_TV                     0
device_Tablet                 0
payment_method_Crypto         0
payment_method_Debit Card     0
payment_method_Gift Card      0
payment_method_PayPal         0
favorite_genre_Comedy         0
favorite_genre_Documentary    0
favorite_genre_Drama          0
favorite_genre_Horror         0
favorite_genre_Romance        0
favorite_genre_Sci-Fi         0
subscription_type_Premium     0
subscription_type_Standard    0
region_Asia                   0
region_Europe                 0
region_North America          0
region_Oceania                0
region_South America          0
dtype: int64
Subscription sum: 1
R

In [142]:
print(new_user_scaled)

[[ 4.61470633e-01  2.36156098e-01 -6.21523800e-02  5.04664405e+01
  -1.42996464e+00 -2.42449306e-02 -7.03080065e-01  1.43460935e+00
  -5.01874304e-01  1.99501370e+00 -4.97811532e-01 -5.14958432e-01
  -4.98437008e-01  1.96325468e+00 -4.92488306e-01 -5.08112348e-01
  -3.98432619e-01 -4.13141649e-01  2.41659787e+00 -4.07819533e-01
  -4.11813845e-01 -4.10151556e-01 -7.15502872e-01  1.42746876e+00
   2.22380377e+00 -4.58011989e-01 -4.52890345e-01 -4.25014758e-01
  -4.59928040e-01]]
